# MPPP — 04 Camera models across scapes (MPPP 0.43.0)

This notebook takes the COLMAP solutions of several scapes, made by notebook 03, and looks closely at the refined camera models: the Navcam left and right cameras and their stereo rig, and the Mastcam-Z 34 mm eyes with focal length as a function of focus.

| section | question | output |
|---|---|---|
| 2 Navcam intrinsics | How much do the refined cameras differ between scapes, and from the shipped models? The differences are measured in pixels over the whole frame, after removing the camera rotation that a pose absorbs. | tables, difference maps, radial profiles |
| 2d–2f Joint calibration (v0p35), rig drift (v0p35.1) | Is the stereo rig stable (temperature, sol, network)? One bundle adjustment of all blocks with shared cameras, thermal focal length and a temperature-dependent rig; which lens model predicts a new block best? | rig tests, joint cameras with covariance, leave-one-out, frozen cameras for notebook 03 |
| 3 Navcam labels | How far is the flight calibration (label CAHVORE) from the refined cameras? | pixel differences |
| 4 Navcam stereo | What do the rig and intrinsics differences do to disparity and range? | disparity bias and spread, range error vs range |
| 5 Mastcam-Z focus | How does focal length depend on focus count across scapes, and how does it compare with the label? | f(focus) fit, scape-to-scape offsets |
| 6 Mastcam-Z pairs | How well does the label (or one fixed rig) predict each left/right pair? | per-pair disparity and range error |
| 7 Performance | What limits the geometry, and by how much? | one table |
| 8 Updated calibration | Refined cameras written back as CAHVORE (Navcam) and CAHVOR (Mastcam-Z) models | `updated_camera_models.json` / `.txt` |
| 9 Example images | Each camera's example image, original and undistorted, with the hardware mask screened in white | PNGs |

**Conventions.** All cameras are full frame: Navcam is 5120 × 3840 and Mastcam-Z is 1648 × 1200. COLMAP pixels have their origin at the corner of the first pixel; CAHV-family models have it at the centre (`mppp.cmod` converts between them). Differences between two cameras are given after the best-fitting rotation has been removed, because a pose absorbs a rotation (a principal-point shift is mostly a rotation). The rotation is reported. Polynomial Navcam models are compared only where they are invertible, which excludes the outer corners (see `coverage`).

**Label models.** The M2020 Navcam labels use CAHVORE type 2 (fisheye). CAHVOR describes them no better than 9 px rms (up to 120 px in the corners), whereas CAHVORE reproduces the refined rational camera to 0.4 px. From v0p21 the MPPP manifest keeps each image's full label model. Older manifests keep only its CAHV part plus R1 and R2, so O is assumed to lie along A; this costs about 1.5 px rms for Navcam and 0.2 px for Mastcam-Z. To read the exact label models from your PDS archive, set `PDS_DIR`.

Figures, tables and the updated models go to `OUT`.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import sys, json, datetime
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import numpy as np
import matplotlib.pyplot as plt
import mppp
from mppp.sfm import calibration as CAL
from mppp.cmod import PixelCamera, compare_cameras
print("mppp", mppp.__version__, "from", Path(mppp.__file__).parent)

try:
    import pandas as pd
    from IPython.display import display
    def show(rows, cols=None, digits=3):
        df = pd.DataFrame(rows)
        if cols:
            df = df[[c for c in cols if c in df]]
        display(df.round(digits))
except ImportError:                                   # plain printout without pandas
    pd = None
    def show(rows, cols=None, digits=3):
        cols = cols or (list(rows[0]) if rows else [])
        print("  ".join(f"{c:>14}" for c in cols))
        for r in rows:
            print("  ".join(f"{r.get(c):>14.{digits}f}" if isinstance(r.get(c), float) else f"{str(r.get(c)):>14}"
                            for c in cols))

def savefig(fig, name):
    p = OUT / name
    fig.savefig(p, dpi=130, bbox_inches="tight")
    print("saved", p)

## 1  Scapes

With `SCAPES = None` (default) every site folder below `SCAPES_ROOT` that notebook 03 finished is used (`mppp.sfm.sites.discover_scapes`), in sol order and labelled from the folder name (`rockytop_colmap` → "Rockytop", `threeforks_south_colmap` → "Three Forks South", `<site>_colmap_nav_zcam34` → "<Site> N+Z34"). A folder is left out when its `project.json` records no finished alignment (a run in progress rewrites it first), when it has fewer than 10 images, or when none of its images falls in the site's sol range in `SITES` (a folder that still holds a renamed site's block). The table below lists every folder and why one was left out. `SCAPE_EXCLUDE` leaves out more; a dict in `SCAPES` analyses just those.

The manifest with the label models is read from `<name>_colmap/processed/`. `PDS_DIR` is optional. When set, the exact label camera models are read from the `.IMG` labels under it. This is only needed for manifests written before v0p21, and a large archive takes a minute or two.


In [ ]:
SCAPES_ROOT = Path(r"D:\scapes\colmap")
SCAPES = None                       # None: every finished alignment below SCAPES_ROOT (mppp.sfm.sites.discover_scapes);
                                    # or a dict {name: WORK folder} to analyse just those
SCAPE_EXCLUDE = []                  # site folders, sites or labels to leave out, e.g. ["whale_mountain"]
INCLUDE_ZCAM34 = True               # also the <site>_colmap_nav_zcam34 blocks (labelled "<Site> N+Z34", Mastcam-Z sections)
SITES = None                        # sol ranges for the folder check (None: mppp.sfm.sites.SITES, as notebook 03)
PDS_DIR = Path("D:/data/m2020")     # the PDS archive: label camera models and camera temperatures (section 2a); None = skip
OUT = SCAPES_ROOT / "camera_analysis" / datetime.date.today().isoformat()

MIN_OBS = 1000                      # a refined camera needs this many observations to be compared
EPS_REF_PX = 0.25                   # reference image precision drawn on the repeatability plot (from notebook 05)
FOCUS_MIN = -1000                   # Mastcam-Z focus fit: bins below this motor count are left out
FOCUS_THERMAL = True                # v0p42: §5b also fits f against the focal-plane temperature (HEAD_FPA, from PDS_DIR)
FOCUS_TREND = True                  # v0p42: §5b also fits f against sol (no per-scape offsets then)
FOCUS_NAVCAM_NORMALISE = True       # v0p42: divide each block's Mastcam-Z f by its Navcam focal scale (refined/start)
RANGES_M = (2, 5, 10, 20, 50)       # ranges for the stereo error tables
SCREEN_ALPHA = 0.3                  # white screen over the hardware mask in the example images
EXAMPLE_SCAPE = {}                  # e.g. {"NL": "Rockytop", "ZL034": "Rockytop N+Z34"} (section 9)
EXAMPLE_FILES = {}                  # e.g. {"NL": (r"D:\...\NLF_....png", r"D:\...\masks\NLF_....png")}
THERMAL_SOURCE = "auto"             # thermal model of the consensus (section 2a): "auto" (within-block bins if
                                    # measured, else across scapes), "within", "across", "fixed" (THERMAL_PPM), None = off
THERMAL_PPM = None                  # ppm/degC for THERMAL_SOURCE = "fixed"
THERMAL_EXPERIMENT = (_root / "docs" / "results" / "v0p31" / "temperature_bins.json") if _root else None
                                    # JSON of scripts/temperature_bins_experiment.py: within-block bins of scapes whose
                                    # alignment has no thermal stage (the nine 0.22.4 scapes; None = only thermal stages)

In [ ]:
from mppp.sfm import sites as SITES_MOD
if SCAPES is None:
    SCAPES = SITES_MOD.discover_scapes(SCAPES_ROOT, sites=SITES, include_zcam=INCLUDE_ZCAM34, exclude=SCAPE_EXCLUDE)
print(len(SCAPES), "scapes:", ", ".join(SCAPES))


In [ ]:
OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)
sols = CAL.load_solutions(SCAPES)
if PDS_DIR:
    for s in sols.values():
        CAL.attach_pds_labels(s, PDS_DIR)
show(CAL.solutions_table(sols))
lenses = {s.navcam_distortion for s in sols.values()} - {"-"}
if "polynomial" in lenses:
    print("Note: some solutions use the polynomial Navcam model (made before v0p20, or with navcam_distortion="
          "'polynomial'): their outer corners are not modelled.")

## 2  Navcam intrinsics across scapes

The first table lists the refined parameters of each scape; `dfx` to `dcy` are the changes from the starting camera. The parameters are strongly correlated (f against k1, and principal point against attitude), so comparing them one by one is misleading. The comparisons that follow therefore map each refined camera against a reference over a pixel grid covering the whole frame. The difference is split into the centre (within half the half-diagonal), the edge (0.5–0.85) and the corners (beyond 0.85).

Two references are used. **Shipped** is the model MPPP starts from (`M2020_N{L,R}_rational.json`). **Consensus** is the observation-weighted mean of the refined cameras with the same lens model. The spread about the consensus is the scape-to-scape repeatability of the calibration.

In [ ]:
show(CAL.camera_table(sols, "N"), ["scape", "camera", "lens", "images", "observations", "fx", "fy", "cx", "cy",
                                   "k1", "k2", "k3", "k4", "p1", "p2", "dfx", "dcx", "dcy"], 5)

### 2a  Focal length against camera temperature

The label camera models are interpolated by the camera-plate temperature (`INTERPOLATION_VALUE`), but their focal length hardly changes with it. The refined focal lengths of the scapes differ by up to about 2 px, in both eyes together. Two measurements of the thermal coefficient are made here:

* **Across scapes**: each scape's refined focal length against its median camera temperature (from the manifest, written by MPPP 0.30 or later, or read from the PDS labels when `PDS_DIR` is set). Anything else that differs between scapes and correlates with the season enters this slope.
* **Within blocks** (v0p31): notebook 03 with `THERMAL_BINS_DEG` splits each eye into cameras by temperature bin and refines fx, fy per bin with the poses, the rest of the camera and the rig shared. The fit is `f = a_scape + b T` per eye (one offset per scape, one slope, weights √observations), so only the temperature changes within a block count. `THERMAL_EXPERIMENT` adds the bins of `scripts/temperature_bins_experiment.py`.

`THERMAL_SOURCE` picks the model the consensus uses: every refined camera's fx, fy are scaled to the reference temperature T0 (the observation-weighted mean temperature) before averaging, and the consensus is written with its thermal entry so that notebook 03 scales it to each block's temperature.

In [ ]:
TEMPS = CAL.camera_temperatures(sols, "N")
show(TEMPS, ["scape", "camera", "images", "n_temp", "temp_median_degC", "temp_min_degC", "temp_max_degC", "fx", "fy"], 3)
TEMPS_SCAPE = CAL.merge_temperature_bins(TEMPS)     # temperature-bin cameras of a scape merged per eye
FT = CAL.focal_temperature_fit(TEMPS_SCAPE)
for cam, f in FT.items():
    print(f"across scapes, {cam}: fx = {f['fx_at_0C']:.2f} px {f['px_per_degC']:+.4f} ± {f['px_per_degC_sd']:.4f} px/degC "
          f"({f['ppm_per_degC']:+.1f} ppm/degC) over {f['scapes']} scapes; corr {f['corr']:+.2f}, residual {f['residual_rms_px']:.2f} px")

from mppp.sfm.thermal import fit_focal_temperature
BIN_ROWS = CAL.thermal_bin_rows(sols, THERMAL_EXPERIMENT)
BIN_FIT = {p: fit_focal_temperature(BIN_ROWS, p, min_observations=MIN_OBS) for p in ("fx", "fy")}
if BIN_ROWS:
    show(BIN_ROWS, ["scape", "camera", "images", "T_median_degC", "T_min_degC", "T_max_degC", "fx", "fy", "observations", "source"], 2)
for p, fit in BIN_FIT.items():
    for eye, f in fit.items():
        print(f"within blocks, {eye} {p}: {f['px_per_degC']:+.4f} ± {f['sd']:.4f} px/degC ({f['ppm_per_degC']:+.1f} ± "
              f"{f['ppm_sd']:.1f} ppm/degC) from {f['rows']} bins of {f['scapes']} scapes; residual {f['residual_rms_px']:.2f} px")
if not BIN_ROWS:
    print("no within-block temperature bins: run notebook 03 with THERMAL_BINS_DEG, or set THERMAL_EXPERIMENT")

THERMAL = None
if THERMAL_SOURCE:
    THERMAL = CAL.thermal_model(sols, BIN_FIT["fx"], FT, THERMAL_SOURCE, THERMAL_PPM, MIN_OBS)
print("thermal model of the consensus:", THERMAL or "none (the cameras are averaged as they are)")

if any(r["n_temp"] for r in TEMPS):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    ax = axes[0]
    for cam, mk in (("NL", "o"), ("NR", "s")):
        rr = [r for r in TEMPS_SCAPE if r["camera"] == cam and r["temp_median_degC"] is not None]
        if not rr:
            continue
        T = np.array([r["temp_median_degC"] for r in rr]); f = np.array([r["fx"] for r in rr])
        lo = T - np.array([r["temp_min_degC"] for r in rr]); hi = np.array([r["temp_max_degC"] for r in rr]) - T
        eb = ax.errorbar(T, f - np.median(f), xerr=[lo, hi], fmt=mk, ms=5, capsize=2, alpha=0.8,
                         label=f"{cam}: {FT[cam]['px_per_degC']:+.3f} px/degC" if cam in FT else cam)
        if cam in FT:
            tt = np.linspace(T.min(), T.max(), 10)
            ax.plot(tt, FT[cam]["fx_at_0C"] + FT[cam]["px_per_degC"] * tt - np.median(f), lw=1, color=eb[0].get_color())
    ax.set_xlabel("camera temperature, median over the scape's images (bars: range) [degC]")
    ax.set_ylabel("refined fx - median over scapes [px]")
    ax.set_title("across scapes (one camera per eye)")
    ax.grid(alpha=0.3); ax.legend()
    ax = axes[1]
    for eye, mk in (("NL", "o"), ("NR", "s")):
        f = BIN_FIT["fx"].get(eye)
        rr = [r for r in BIN_ROWS if r["eye"] == eye and f and r["scape"] in f["offsets"] and r["observations"] >= MIN_OBS]
        if not rr:
            continue
        T = np.array([r["T_median_degC"] for r in rr])
        d = np.array([r["fx"] - f["offsets"][r["scape"]] for r in rr])
        sc = ax.scatter(T, d - f["px_per_degC"] * np.mean(T), marker=mk, s=12 + 60 * np.sqrt(np.array([r["observations"] for r in rr]) / 1e6),
                        alpha=0.7, label=f"{eye}: {f['px_per_degC']:+.3f} ± {f['sd']:.3f} px/degC")
        tt = np.linspace(T.min(), T.max(), 10)
        ax.plot(tt, f["px_per_degC"] * (tt - np.mean(T)), lw=1, color=sc.get_facecolor()[0][:3])
    if THERMAL:
        for eye, m in THERMAL.items():
            ax.axvline(m["T0_degC"], color="k", lw=0.6, ls=":")
    ax.set_xlabel("bin temperature, median [degC]")
    ax.set_ylabel("bin fx - scape offset [px]")
    ax.set_title("within blocks (temperature bins; dotted: T0)")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "navcam_focal_vs_temperature.png")
else:
    print("no camera temperatures: set PDS_DIR (the labels are read) or process the images with MPPP 0.30 or later")

### 2b  Consensus and differences

The consensus of each eye (with `THERMAL`: at T0) and the pixel difference of every refined camera from the shipped model and from the consensus. With a thermal model the consensus is first scaled to each camera's temperature, so the difference is what remains after the temperature correction.

In [ ]:
NAV = {}
for g in ("NL", "NR"):
    for lens in sorted(lenses):
        c = CAL.consensus_camera(sols, g, lens, MIN_OBS, thermal=THERMAL)
        if c is not None:
            NAV[(g, lens)] = c
            print(f"{c.key}: " + ", ".join(f"{k} {v:.6g}" for k, v in c.named().items() if k not in ("k5", "k6"))
                  + (f"  (at T0 {c.thermal['T0_degC']:.1f} degC)" if getattr(c, "thermal", None) else ""))

DIFF = []
for g in ("NL", "NR"):
    ship = CAL.reference_camera(g, "rational")
    for r in CAL.reference_differences(sols, g, ship, min_observations=MIN_OBS):
        DIFF.append(dict(r, camera=g, key=r["camera"], reference="shipped rational"))
    for lens in sorted(lenses):
        if (g, lens) in NAV:
            for r in CAL.reference_differences(sols, g, NAV[(g, lens)], min_observations=MIN_OBS, thermal=THERMAL):
                if r["lens"] == lens:
                    DIFF.append(dict(r, camera=g, key=r["camera"], reference=f"consensus {lens}"))
show(DIFF, ["camera", "reference", "scape", "key", "lens", "T_degC", "rms_px", "centre_rms_px", "edge_rms_px", "corner_rms_px",
            "max_px", "coverage", "rotation_deg"], 3)

In [ ]:
# repeatability: the spread about the consensus, per camera and lens model
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for ax, g in zip(axes, ("NL", "NR")):
    rows = [r for r in DIFF if r["camera"] == g and r["reference"].startswith("consensus")]
    x = np.arange(len(rows))
    for k, (key, lab) in enumerate((("centre_rms_px", "centre"), ("edge_rms_px", "edge"), ("corner_rms_px", "corners"))):
        ax.bar(x + (k - 1) * 0.27, [r[key] for r in rows], 0.27, label=lab)
    ax.set_xticks(x, [f"{r['scape']}{' ' + r['key'][3:] if r['key'] != g else ''}\n({r['lens'][:4]})" for r in rows], rotation=45, ha="right", fontsize=8)
    ax.set_yscale("log")
    ax.axhline(EPS_REF_PX, color="k", lw=0.8, ls="--")
    ax.text(len(rows) - 0.5, EPS_REF_PX * 1.04, f"ε ≈ {EPS_REF_PX:g} px (EPS_REF_PX)", ha="right", va="bottom", fontsize=8)
    ax.set_title(f"{g}: difference from the consensus of its lens model (rotation removed)")
    ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("rms pixel difference")
axes[0].legend()
fig.tight_layout()
savefig(fig, "navcam_repeatability.png")

### 2c  Verified consensus cameras for notebook 03

The consensus rational Navcam cameras and the mean rig are written in the format of the shipped start cameras, with each scape's rms distance from the consensus recorded as `verification`. Point notebook 03's `NAVCAM_CAMERAS` at this folder to start projects from them (`SfmProject.create(navcam_cameras=...)`); with `NAVCAM_INTRINSICS = "hold"` (or `"auto"` on a weak network) they are the cameras of the solution.

In [ ]:
CONSENSUS_DIR = OUT / "navcam_consensus"
if ("NL", "rational") in NAV and ("NR", "rational") in NAV:
    rig_c = CAL.mean_rig(sols, "rational")
    rep_rows = [{k: r[k] for k in ("camera", "scape", "rms_px", "centre_rms_px", "edge_rms_px", "corner_rms_px", "rotation_deg")}
                for r in DIFF if r["reference"] == "consensus rational"]
    written = CAL.write_navcam_consensus({"NL": NAV[("NL", "rational")], "NR": NAV[("NR", "rational")]}, rig_c, CONSENSUS_DIR,
                                         sols=sols, repeatability={"per_scape": rep_rows, "eps_ref_px": EPS_REF_PX})
    for k, f in written.items():
        print(k, "->", f)
    v = json.loads(written["NL"].read_text(encoding="utf-8"))["verification"]
    print("NL: scapes within the consensus:", [(r["scape"], round(r["rms_px"], 3)) for r in v["per_scape"] if not r["excluded"]],
          "| excluded:", [(e["scape"], round(e["rms_px"], 2)) for e in v["excluded"]] or "none")
else:
    print("no rational consensus: nothing written")

### 2d  Joint calibration, rig stability and lens model (v0p35)

Sections 2b–2c average cameras that each block refined on its own. Here all blocks go into **one bundle adjustment**
(`scripts/navcam_calibration_study.py`, `mppp.sfm.navcal`), and the stereo rig is tested for stability:

* **Rig stability.** Each block is re-adjusted (at most `RIG_POINTS` points) with the right camera's rotation in the rig free, (i) as the pipeline does, (ii) with both principal points held at values common to all blocks — the rig yaw and the principal-point difference both shift the disparity at infinity and trade against each other, so only then is the yaw comparable between blocks — (iii) with the right camera's position in the rig free as well (baseline length and direction), and (iv) with one rig per 10 °C bin. Every rig has its covariance (3-D points eliminated). The rigs are tested against camera temperature, the left–right temperature difference, sol and network strength, between blocks (weighted, with a between-block scatter τ) and within blocks (per-bin rigs, one offset per block).
* **Joint calibration.** One camera per eye, one rig and each block's own poses and points (`JOINT_POINTS` points per block). The focal length follows f(T) = f0 (1 + b (T − T0)) and the rig yaw turns by k (T − T0); both enter through the keypoints (`bundle_adjust(keypoint_scale=..., keypoint_map=...)`), and b and k are profiled (cost minimum over a grid). The shared cameras and rig carry their covariance.
* **Lens model.** Leave one block out: the joint calibration of the other blocks predicts the held-out block's cameras and rig. The rational model (`FULL_OPENCV`, k4 free) and the fisheye + tangential model (`THIN_PRISM_FISHEYE`, sx1 = sy1 = 0) are compared on the held-out fit and on the distance of the prediction from the block's own calibration.

The study takes a few hours and about 5 GB of memory; run it from a terminal (`python scripts/navcam_calibration_study.py all STUDY_DIR scapes.json`) or with `RUN_STUDY = True`. This section reads its results.

In [ ]:
from mppp.sfm import navcal as NC
from mppp.sfm import navcal_report as NCR
STUDY_DIR = SCAPES_ROOT / "camera_analysis" / "navcal_v0p40"   # OUT_DIR of scripts/navcam_calibration_study.py all
                                                                  # (v0p40: 23 blocks; v0p35: navcal_v0p35, 15 blocks)
LABEL_SAMPLES = None       # JSON {PDS stem: {"NL", "NR"}} of label temperatures, for blocks whose project records none
                           # (projects made before MPPP 0.31); None: the project records only
RUN_STUDY = False          # True: run the three studies here (hours)
JOINT_POINTS = 15000       # points per block in the joint calibration
RIG_POINTS = 120000        # at most this many points per block in the rig study

STUDY_DIR = Path(STUDY_DIR)
cfg = STUDY_DIR / "scapes.json"
if RUN_STUDY:
    import subprocess
    STUDY_DIR.mkdir(parents=True, exist_ok=True)
    blocks = {k: str(v) for k, v in SCAPES.items() if (Path(v) / "colmap" / "sparse" / "cahv_ba").is_dir()}
    cfg.write_text(json.dumps(blocks, indent=1), encoding="utf-8")
    cmd = [sys.executable, str(_root / "scripts" / "navcam_calibration_study.py"), "all", str(STUDY_DIR), str(cfg),
           "--points", str(JOINT_POINTS), "--rig-points", str(RIG_POINTS)] + (["--samples", str(LABEL_SAMPLES)] if LABEL_SAMPLES else [])
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
REPORT_DIR = OUT / "navcal"
RES, COMP = NCR.main(STUDY_DIR / "rig", STUDY_DIR / "joint", REPORT_DIR, cfg if cfg.is_file() else None, LABEL_SAMPLES)

In [ ]:
# rig per block: pipeline rig (yaw trades with the principal points), common principal points, free baseline, per-bin rigs
RIG_T = json.loads((REPORT_DIR / "rig_study.json").read_text())
show(RIG_T, ["scape", "stations", "span_m", "T_median_degC", "dT_LR_degC", "sol", "rotation_yaw_mdeg", "rotation_pp_yaw_mdeg",
             "rotation_pp_yaw_sd_mdeg", "rotation_pp_pitch_mdeg", "rotation_pp_roll_mdeg", "rotation_disparity_inf_px",
             "full_baseline_m", "full_baseline_sd_mm"], 3)
RT = json.loads((REPORT_DIR / "rig_tests.json").read_text())
rows = []
for ang in ("yaw", "pitch", "roll"):
    t = RT[f"rotation_pp:{ang}"]; w = t.get("within") or {}
    row = {"angle": ang, "mean_mdeg": t["mean_mdeg"], "scatter_sd_mdeg": t["scatter_sd_mdeg"], "formal_sd_mdeg": t["median_sd_mdeg"],
           "tau_mdeg": t["tau_mdeg"], "within_slope_mdeg_per_degC": w.get("slope_mdeg_per_degC"), "within_se": w.get("se")}
    for c in ("T_median_degC", "dT_LR_median_degC", "sol_median", "observations"):
        v = t["covariates"].get(c, {})
        row[f"{c}: slope"] = v.get("slope"); row[f"{c}: p"] = v.get("p")
    rows.append(row)
show(rows, None, 4)

In [ ]:
# the joint calibration: thermal slopes, shared cameras (at T0) with standard deviations, rig
for lens in ("rational", "fisheye_t"):
    j = RES.get(f"joint_{lens}")
    if not j:
        continue
    th = j.get("rig_thermal") or {}
    print(f"{lens}: T0 {j['T0_degC']:.1f} degC, focal {j['ppm_per_degC']:.1f} +- {j['sd_ppm_per_degC']:.1f} ppm/degC, rig yaw "
          f"{th.get('yaw_mdeg_per_degC', 0):+.3f} +- {(th.get('yaw_profile') or {}).get('sd', float('nan')):.3f} mdeg/degC, "
          f"variance factor {j['variance_factor']:.3f}, {j['final']['observations']} observations of {len(j['blocks'])} blocks")
    show([{"camera": g, **{k: j[g]["params"][k] for k in list(j[g]["params"])[:10]}} for g in ("NL", "NR")], None, 5)
    show([{"camera": g, **(j[g].get("sd") or {})} for g in ("NL", "NR")], None, 5)
    if j.get("profile"):
        show(j["profile"], ["ppm_per_degC", "cost", "iterations"], 2)
    if (th.get("yaw_profile") or {}).get("rows"):
        show(th["yaw_profile"]["rows"], ["yaw_mdeg_per_degC", "cost", "iterations"], 3)

In [ ]:
# leave one block out: held-out fit (cost increase over the block's own calibration) and camera prediction error
for lens in ("rational", "fisheye_t"):
    if lens in COMP:
        print(lens, {k: round(v, 3) for k, v in COMP[lens]["summary"].items()})
        show(COMP[lens]["rows"], ["scape", "cost_cameras_pct", "cost_all_pct", "cost_no_rigT_pct", "held_median_px", "own_median_px",
                                  "held_corner_px", "own_corner_px", "pred_rms_NL", "pred_rms_NR", "dfx_NL", "dfx_NR",
                                  "disparity_inf_diff_px"], 3)
if "paired" in COMP:
    print("fisheye + tangential minus rational:", json.dumps({k: v for k, v in COMP["paired"].items() if k != "per_block"}, indent=1))
from IPython.display import Image as _Img, display as _disp
_disp(_Img(filename=str(REPORT_DIR / "navcam_calibration.png")))

### 2f  Rig drift with more blocks (v0p35.1; sol epochs v0p40)

Is the rig's change over the mission (section 2d) robust? The rig study is repeated on blocks the joint calibration did not use (`python scripts/navcam_calibration_study.py rig DRIFT_DIR/rig scapes.json --samples label_temps.json --common-pp STUDY_DIR/rig/common_principal_points.json`, the v0p35 rig files copied into `DRIFT_DIR/rig` first). Blocks solved with thermal bins are read from their one-camera-per-eye model (`sparse/cahv_ba_single`), blocks solved with the fisheye lens are refitted to the rational model, and every block's angles are referred to one label rig (the label calibration changed once, near sol 250). Then:

* the drift fit on all usable blocks, with a faster early rate for pitch before sol 300 (`NC.rig_drift_model`: rate × (sol − sol0) + (early rate − rate) × (min(sol, 300) − ref));
* the fit on the v0p35 blocks only, predicting the new ones (residual z = residual / prediction sd, with τ);
* leave-one-block-out rates (jackknife), the fits with sol alone and temperature alone, and the leave-one-block-out prediction of each block's rig by four models (one rig; temperature; temperature + sol; + the pitch hinge).

`DRIFT` is what section 2e writes into `M2020_N_rig.json`; blocks in `EXCLUDE` are shown but not fitted.

**Sol epochs (v0p40).** Nearby waypoints can bring images of another epoch into a block: Sid holds sols 91–101 and 360–371, on either side of the pitch knee at sol 300, and one rig at the block's median sol mixes the two. The rig study's `epochs` mode (`navcam_calibration_study.py rig ... --modes epochs`, `NC.split_rig_by_epoch`: one rig per group of sols more than 30 sols from the next, cameras shared, common principal points) solves one rig per epoch, and with `EPOCHS = True` each epoch with at least 6 stereo frames enters the fits at its own sol and temperature (`NC.expand_epochs`; Sid, Three Forks South, South Arm in the v0p40 study).


In [ ]:
DRIFT_DIR = SCAPES_ROOT / "camera_analysis" / "navcal_v0p40"   # rig/ of the extended study (v0p40: 23 blocks)
NEW_BLOCKS = ["Overlook Mountain", "Olifants", "Groloy"]          # added in v0p40 (predicted by the fit without them)
EXCLUDE = ["Whale Mountain"]           # weak network (2 stations, cameras and rig held in notebook 03): shown, not fitted
EPOCHS = True                          # v0p40: blocks with images of two sol epochs enter the fits once per epoch
rig_dir = DRIFT_DIR / "rig" if (DRIFT_DIR / "rig").is_dir() else STUDY_DIR / "rig"
ST_BLOCKS = NCR.load_rig(rig_dir)      # angles against one label reference (NC.common_reference)
ST = NC.expand_epochs(ST_BLOCKS) if EPOCHS else ST_BLOCKS
SPLIT = [s["scape"] for s in ST if s.get("parent")]
if SPLIT:
    print("fitted per sol epoch:", ", ".join(SPLIT))
_jb = list(((RES.get("joint_fisheye_t") or RES.get("joint_rational") or {}).get("blocks") or {}))
_ref = [s for s in ST if s.get("parent", s["scape"]) in _jb] or None   # the joint rig's blocks: the drift averages to 0 over them
DRIFT = NC.rig_drift_model([s for s in ST if s["scape"] not in EXCLUDE], reference=_ref)
ROB = NC.drift_robustness(ST, exclude=EXCLUDE, holdout=[s["scape"] for s in ST if s.get("parent", s["scape"]) in NEW_BLOCKS
                                                       and s["scape"] not in EXCLUDE])
print(f"{DRIFT['blocks']} blocks, sols {DRIFT['sol_range'][0]:.0f}-{DRIFT['sol_range'][1]:.0f}, sol0 {DRIFT['sol0']:.0f}")
show([{"angle": a, "rate_mdeg_per_1000sol": 1e3 * DRIFT[f"{a}_fit_mdeg_per_sol"], "se": 1e3 * DRIFT[f"{a}_se"],
       "p": DRIFT[f"{a}_p"], "used_rate": 1e3 * DRIFT[f"{a}_mdeg_per_sol"],
       "early_rate_before_knot": 1e3 * DRIFT[f"{a}_early_mdeg_per_sol"] if f"{a}_early_mdeg_per_sol" in DRIFT else None,
       "T_mdeg_per_degC": DRIFT[f"{a}_T_mdeg_per_degC"], "tau_mdeg": DRIFT[f"{a}_tau_mdeg"],
       "jackknife_min": 1e3 * ROB[f"{a}_jackknife"]["min"], "jackknife_max": 1e3 * ROB[f"{a}_jackknife"]["max"],
       "sign_stable": ROB[f"{a}_jackknife"]["sign_stable"]} for a in ("pitch", "yaw", "roll")], None, 3)
if ROB.get("holdout"):
    print("fit without the new blocks, predicting them (mdeg):")
    show([{"scape": p["scape"], "sol": p["sol"], **{f"{a}_{k}": p[a][k] for a in ("pitch", "yaw", "roll")
                                                   for k in ("residual_mdeg", "sd_pred_mdeg", "z")}}
          for p in ROB["holdout"]["predictions"]], None, 2)
show([{"angle": a, **{k: v["rms_mdeg"] for k, v in d.items()}} for a, d in ROB["loo_prediction"].items()], None, 2)
_old = STUDY_DIR.parent / "navcal_v0p35p1" / "navcam_joint" / "M2020_N_rig.json"     # the previous (v0p35.1) drift
_old = json.loads(_old.read_text()).get("drift") if _old.is_file() else NC.rig_drift(NCR.load_rig(STUDY_DIR / "rig"))
NCR.drift_figure(ST, DRIFT, OUT / "navcam_rig_drift.png", highlight=SPLIT + NEW_BLOCKS, old_drift=_old,
                 labels={"fit": "v0p40 fit", "old": "v0p35.1 drift", "base": "blocks",
                         "new": "new block or sol epoch"})
_disp(_Img(filename=str(OUT / "navcam_rig_drift.png")))


### 2g  Pixel offsets between the Navcam resolutions (v0p35.1)

Navcam products come at full (5120 × 3840), half and quarter resolution. MPPP maps native keypoints to full-frame pixels by a pure scale in COLMAP's corner-origin pixels (x_full = x / scale; with pixel centres, x_full = (x + 0.5) / scale − 0.5): exact for binning that starts at the detector corner. A different relation (e.g. sub-sampling) would put the half- and quarter-resolution images 0.5 and 1.5 px off the full-resolution ones, in x and y alike and in both eyes. Two checks:

* **Labels.** The flight camera model of every product, mapped to full-frame pixels the same way, per eye and sol against the half-resolution products (`NCR.label_scale_consistency`).
* **Images.** Each block re-adjusted with one camera per eye and scale (`NC.scale_offsets`: `split_by_scale`, principal point free — `pp` — or principal point and focal length — `ppf` —, everything else and the rig held, with covariance; `scripts/navcam_calibration_study.py scale DRIFT_DIR/scale scapes.json`). Offsets are (scale) − (half resolution) in full-resolution px. What stereo sees is the left-minus-right difference: a common shift of both eyes is an attitude change of those frames.


In [ ]:
import glob as _glob
LBL = NCR.label_scale_consistency(sorted({str(p) for v in SCAPES.values()
                                           for p in (Path(v).parent if Path(v).name == "colmap" else Path(v)).glob("processed/mppp_manifest_v0p*.json")}))
print(LBL["products"], "Navcam products with a label model")
show([{"pair": k, **{kk: vv for kk, vv in v.items() if not kk.endswith("iqr_px")}} for k, v in LBL["differences"].items()], None, 3)
SCALE_DIR = DRIFT_DIR / "scale"
if SCALE_DIR.is_dir():
    for tag in ("pp", "ppf"):
        SS = NCR.scale_summary(SCALE_DIR, tag)
        print(f"{tag}: left - right", {k: round(v, 3) for k, v in SS["left_minus_right"].items()})
        show([{"pair": k, **v} for k, v in SS["pairs"].items()], None, 3)
    show(NCR.scale_summary(SCALE_DIR, "ppf")["rows"], None, 3)
else:
    print("no scale study in", SCALE_DIR)


### 2e  Frozen Navcam cameras for notebook 03 (v0p35)

The joint calibration of the chosen lens model is written in the format of the shipped start cameras (`M2020_NL_fisheye_tangential.json` / `M2020_NR_fisheye_tangential.json` for the fisheye + tangential model, notebook 03 `NAVCAM_DISTORTION = "fisheye_tangential"`; `M2020_NL_rational.json` / `M2020_NR_rational.json` for the rational one; and `M2020_N_rig.json`) with the thermal slope of the focal length, the rig's temperature slope, the covariance and the leave-one-out verification. Point notebook 03's `NAVCAM_CAMERAS` at `JOINT_DIR`: `SfmProject.create` scales fx, fy to the block's median camera temperature and turns the rig to it; the thermal stage turns each temperature bin's rig by the slope.

In [ ]:
LENS = "fisheye_t"                    # v0p35: fisheye + tangential (better held-out fit in all 15 blocks, same transfer);
                                      # "rational" writes the rational joint calibration instead
JOINT_DIR = OUT / ("navcam_joint" if LENS == "fisheye_t" else "navcam_joint_rational")   # notebook 03: NAVCAM_CAMERAS
# DRIFT: section 2f (v0p35.1: 20 blocks, pitch rate faster before sol 300); v0p35 used NC.rig_drift(NCR.load_rig(STUDY_DIR / "rig"))
print("rig drift per sol:", {k: round(v, 5) for k, v in DRIFT.items() if k.endswith(("_per_sol", "_early_mdeg_per_sol", "sol0"))})
if RES.get(f"joint_{LENS}"):
    written = NC.write_joint_cameras(dict(RES[f"joint_{LENS}"], rig_drift=DRIFT), JOINT_DIR, loo=RES.get(f"loo_{LENS}"),
                                     rig_test={k: {kk: vv for kk, vv in v.items() if kk not in ("rows",)}
                                               for k, v in RT.items() if k.startswith("rotation_pp")})
    for k, f in written.items():
        print(k, "->", f)
else:
    print("no joint calibration results in", STUDY_DIR)

In [ ]:
# difference maps: each scape's NL against the shipped rational model (colour = |difference| in px;
# white = not compared: the polynomial model is not invertible there)
rows = [r for r in DIFF if r["camera"] == "NL" and r["reference"] == "shipped rational"]
n = max(1, len(rows))
cols = min(4, n)
# v0p31: one colour scale for every panel (the 98th percentile over all scapes), and each map drawn as an image
# of its sample grid (no overlapping square markers: those aliased into moire when the figure was scaled)
allv = np.concatenate([r["_diff"]["norm_px"][np.isfinite(r["_diff"]["norm_px"])] for r in rows]) if rows else np.array([1.0])
VMAX = float(np.percentile(allv, 98)) if allv.size else 1.0


def _grid(uv, v):
    xs, ys = np.unique(np.round(uv[:, 0], 3)), np.unique(np.round(uv[:, 1], 3))
    G = np.full((ys.size, xs.size), np.nan)
    G[np.searchsorted(ys, np.round(uv[:, 1], 3)), np.searchsorted(xs, np.round(uv[:, 0], 3))] = v
    dx = (xs[-1] - xs[0]) / max(1, xs.size - 1)
    dy = (ys[-1] - ys[0]) / max(1, ys.size - 1)
    return G, (xs[0] - dx / 2, xs[-1] + dx / 2, ys[-1] + dy / 2, ys[0] - dy / 2)


fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(4.2 * cols, 3.3 * np.ceil(n / cols)), squeeze=False)
im = None
for ax, r in zip(axes.ravel(), rows):
    d = r["_diff"]
    G, ext = _grid(np.asarray(d["uv"], float), np.asarray(d["norm_px"], float))
    im = ax.imshow(G, extent=ext, origin="upper", cmap="viridis", vmin=0.0, vmax=VMAX, interpolation="bilinear")
    ax.set_xlim(0, 5120); ax.set_ylim(3840, 0); ax.set_aspect("equal")
    ax.set_title(f"{r['scape']}{' ' + r['key'][3:] if r['key'] != 'NL' else ''} ({r['lens']})\nrms {r['rms_px']:.2f} px, centre {r['centre_rms_px']:.2f}, "
                 f"corners {r['corner_rms_px']:.1f} px", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.ravel()[n:]:
    ax.set_axis_off()
fig.suptitle("NL: refined camera minus the shipped rational model, rotation removed")
fig.tight_layout(rect=(0, 0, 0.92, 1))
if im is not None:
    cax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
    fig.colorbar(im, cax=cax, label=f"|difference| [px] (0 - {VMAX:.2f}, all panels)")
savefig(fig, "navcam_difference_maps.png")

In [ ]:
# radial profiles: image radius against field angle towards the far corner, as the departure from an
# equidistant fisheye r = f theta (a pinhole would be f tan theta: far off-scale)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, g in zip(axes, ("NL", "NR")):
    for n_, s in sols.items():
        c = max(s.cameras_of(g), key=lambda c: c.n_obs, default=None)     # temperature bins: the most observed
        if c is None or c.n_obs < MIN_OBS:
            continue
        rp = CAL.radial_profile(c)
        ax.plot(rp["theta_deg"], rp["r_px"] - rp["fisheye_px"], lw=1,
                ls="-" if c.distortion == "rational" else "--", label=f"{n_} ({c.distortion[:4]})")
    ship = CAL.reference_camera(g, "rational")
    rp = CAL.radial_profile(ship)
    ax.plot(rp["theta_deg"], rp["r_px"] - rp["fisheye_px"], "k", lw=2, alpha=0.5, label="shipped rational")
    lab = next((m for m in (CAL.median_label_model(s, g) for s in sols.values() if s.manifest) if m is not None), None)
    if lab is not None:
        pc = PixelCamera.cahv(lab, 5120, 3840, as_is=True)
        _, li = lab.decompose()
        f = np.sqrt(li["hs"] * li["vs"])
        th = np.radians(np.linspace(0, 62, 120))
        d = np.array([5120 - li["hc"], 3840 - li["vc"]]); d /= np.linalg.norm(d)
        uv = pc.project(np.c_[np.sin(th) * d[0], np.sin(th) * d[1], np.cos(th)])
        ax.plot(np.degrees(th), np.linalg.norm(uv - 0.5 - [li["hc"], li["vc"]], axis=1) - f * th, "r:", lw=2,
                label="label CAHVORE")
    corner = ship.pixel_camera().rays(np.array([[5120.0, 3840.0]]))[0]
    ax.axvline(np.degrees(np.arccos(corner[2])), color="0.5", lw=0.8)
    ax.text(np.degrees(np.arccos(corner[2])), 2, " frame corner", fontsize=7, color="0.4")
    ax.set_xlabel("field angle towards the corner [deg]")
    ax.set_title(f"{g}: image radius r(θ) − f θ (dashed polynomials turn back where they stop being invertible)",
                 fontsize=9)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("px")
axes[0].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "navcam_radial_profiles.png")

## 3  Navcam: label (flight) calibration vs the refined cameras

The label models vary a little from image to image because they are interpolated by temperature. The first table gives their spread for each scape. The comparison then uses the median label model against the refined consensus and against each scape's refined camera, again after removing the rotation.

In [ ]:
LAB = [r for s in sols.values() for r in CAL.label_model_summary(s) if r["group"] in ("NL", "NR")]
show(LAB, ["scape", "group", "images", "kind", "hs_median", "hs_spread", "hc_median", "vc_median", "hc_spread",
           "vc_spread", "r1_median", "r2_median"], 4)

LABEL_VS = []
for g in ("NL", "NR"):
    lab = next((m for m in (CAL.median_label_model(s, g) for s in sols.values() if s.manifest) if m is not None), None)
    if lab is None:
        print("no manifest with label models"); break
    lp = PixelCamera.cahv(lab, 5120, 3840, as_is=True)
    targets = [(k[1] + " consensus", c) for k, c in NAV.items() if k[0] == g]
    targets += [(n_, s.cameras[g]) for n_, s in sols.items() if g in s.cameras and s.cameras[g].n_obs >= MIN_OBS]
    for name, c in targets:
        d = compare_cameras(lp, c.pixel_camera(), step=96)
        LABEL_VS.append({"camera": g, "refined": name, "lens": c.distortion, **{k: d[k] for k in (
            "rms_px", "centre_rms_px", "edge_rms_px", "corner_rms_px", "max_px", "coverage", "rotation_deg")}})
show(LABEL_VS, None, 3)

## 4  Navcam stereo geometry

The **rig** is the pose of NR relative to NL. The first table gives, for each scape, the refined rotation minus the label rotation as yaw about the left camera's y axis (which moves disparity), pitch about x (vertical parallax) and roll. The baseline stays at the CAHV value, because notebook 03 refines only the rig rotation.

Intrinsics and rig act together, so the useful measure is `stereo_effect`: the disparity error made by processing the pair with geometry **a** when **b** (the refined one) is true. It is evaluated on a grid of image points at each range, with the left camera's absorbable rotation removed. Disparity is d = x_left − x_right. A positive error means **a** puts points too far away, with dZ/Z = Z·Δd/(f·B); for Navcam, f·B ≈ 1254 px·m.

In [ ]:
RIG = []
for n_, s in sols.items():
    pr = CAL.stereo_pairs(s, "N")
    if not pr:
        continue
    a = np.array([[p["dyaw_mdeg"], p["dpitch_mdeg"], p["droll_mdeg"], p["baseline_m"]] for p in pr])
    RIG.append({"scape": n_, "lens": s.navcam_distortion, "pairs": len(pr), "dyaw_mdeg": np.median(a[:, 0]),
                "dpitch_mdeg": np.median(a[:, 1]), "droll_mdeg": np.median(a[:, 2]), "baseline_m": np.median(a[:, 3])})
show(RIG, None, 4)

STEREO = []
for n_, s in sols.items():
    b = CAL.navcam_stereo(s, "refined")
    if b is None:
        continue
    for which in ("label", "initial"):
        a = CAL.navcam_stereo(s, which)
        if a is not None:
            for r in CAL.stereo_effect(*a, *b, ranges_m=RANGES_M):
                STEREO.append({"scape": n_, "lens": s.navcam_distortion, "a": which, **r})
    lens = s.navcam_distortion
    if ("NL", lens) in NAV and ("NR", lens) in NAV:
        rig = CAL.mean_rig(sols, lens)
        a = (NAV[("NL", lens)].pixel_camera(), NAV[("NR", lens)].pixel_camera(), rig)
        for r in CAL.stereo_effect(*a, *b, ranges_m=RANGES_M):
            STEREO.append({"scape": n_, "lens": lens, "a": "consensus", **r})
show([r for r in STEREO if r["range_m"] == 10], ["scape", "lens", "a", "coverage", "disparity_bias_px", "disparity_sd_px",
                                                 "disparity_max_px", "vertical_parallax_rms_px", "centre_disparity_px",
                                                 "range_error_centre_pct", "range_error_centre_m"], 3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3), sharey=True)
for ax, which, title in zip(axes, ("label", "initial", "consensus"),
                            ("label CAHVORE + label rig", "start cameras + label rig", "consensus cameras + mean rig")):
    for n_ in sols:
        rr = [r for r in STEREO if r["scape"] == n_ and r["a"] == which]
        if rr:
            ax.plot([r["range_m"] for r in rr], [r["range_error_centre_pct"] for r in rr], marker="o", ms=3,
                    ls="-" if rr[0]["lens"] == "rational" else "--", label=n_)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xscale("log")
    ax.set_xlabel("range [m]")
    ax.set_title(f"Navcam range error at the image centre\nusing {title}", fontsize=9)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("dZ / Z [%]  (+ = too far)")
axes[0].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "navcam_stereo_range_error.png")

## 5  Mastcam-Z 34 mm: focal length against focus

In notebook 03 every Mastcam-Z eye is split into focus bins. With the default `zcam_bin_refine="focal"`, only the bin's focal length is refined; principal point and distortion are held at the median of the bin's label models. The table therefore describes focal length (and fx/fy) only.

`fit_focus_model` fits f = f0 + a·(focus − ref) across scapes, with a constant per scape, to the bins at or above `FOCUS_MIN` motor counts (default −2000; the few bins below scatter far from the line). The **scape offsets** measure how well the focal length repeats at the same focus. The grey points are the per-image label focal lengths, and the dashed line is their fit. The two eyes share both axes; the right-hand axis gives the focal length in mm for the 7.4 µm pixels.

In [ ]:
FOCUS = CAL.focus_table(sols)
if not FOCUS:
    print("no Mastcam-Z cameras in these scapes")
else:
    FITS = {g: CAL.fit_focus_model(FOCUS, g, min_observations=MIN_OBS, min_focus=FOCUS_MIN)
            for g in sorted({r["group"] for r in FOCUS})}
    show([{k: v for k, v in f.items() if k not in ("scape_offsets_px", "label")} | (
          {"label_f0_px": f["label"]["f0_px"], "label_slope": f["label"]["slope_px_per_count"]} if f["label"] else {})
          for f in FITS.values() if f], None, 4)
    for g, f in FITS.items():
        if f:
            print(g, "scape offsets [px]:", {k: round(v, 2) for k, v in f["scape_offsets_px"].items()})

    # both eyes on the same axes; right-hand axis in mm (7.4 um pixels)
    fig, axes = plt.subplots(1, len(FITS), figsize=(7.5 * len(FITS), 5), squeeze=False, sharex=True, sharey=True)
    for ax, (g, f) in zip(axes[0], FITS.items()):
        ax.axvspan(-1e5, FOCUS_MIN, color="0.92", zorder=0, label=f"left out of the fit (< {FOCUS_MIN})")
        for n_, s in sols.items():
            imgs = [r for r in s.images.values() if s.cameras[r["instrument"]].group == g
                    and r.get("focus_count") is not None and r.get("label_f_px") is not None]
            ax.plot([r["focus_count"] for r in imgs], [r["label_f_px"] for r in imgs], ".", color="0.7", ms=3)
        for k, n_ in enumerate(sorted({r["scape"] for r in FOCUS})):
            rr = [r for r in FOCUS if r["group"] == g and r["scape"] == n_ and r["refined"]]
            good = [r for r in rr if r["observations"] >= MIN_OBS]
            ax.scatter([r["focus"] for r in good], [r["f_refined_px"] for r in good],
                       s=[4 + 0.4 * np.sqrt(r["observations"]) for r in good], color=f"C{k}", label=n_, zorder=3)
            weak = [r for r in rr if r["observations"] < MIN_OBS]
            ax.plot([r["focus"] for r in weak], [r["f_refined_px"] for r in weak], "x", color=f"C{k}", ms=4)
        if f:
            xs = np.linspace(*f["focus_range"], 50)
            ax.plot(xs, f["f0_px"] + f["slope_px_per_count"] * (xs - f["reference_focus"]), "k-", lw=1.2,
                    label=f"fit: {f['slope_px_per_count']:+.4f} px/count ({f['slope_px_per_count'] * CAL.ZCAM_PIXEL_MM * 1e3:+.3f} mm"
                          f"/1000), rms {f['rms_px']:.1f} px")
            if f["label"]:
                ax.plot(xs, f["label"]["f0_px"] + f["label"]["slope_px_per_count"] * (xs - f["reference_focus"]),
                        "k--", lw=1, label="label fit")
        ax.set_xlabel("focus motor count")
        ax2 = ax.secondary_yaxis("right", functions=(lambda v: v * CAL.ZCAM_PIXEL_MM, lambda v: v / CAL.ZCAM_PIXEL_MM))
        ax2.set_ylabel("focal length [mm], 7.4 µm pixels")
        ax.set_title(f"{g}: refined bins (x = < {MIN_OBS} observations), labels in grey")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7, loc="upper left")
    sel = [r for r in FOCUS if r["focus"] is not None and r["focus"] >= FOCUS_MIN]   # weak bins can be far off
    vals = [r["f_refined_px"] for r in sel if r["refined"] and r["observations"] >= MIN_OBS]
    vals += [r["f_label_median_px"] for r in sel if r.get("f_label_median_px")]
    if vals:
        lo, hi = min(vals), max(vals)
        axes[0][0].set_ylim(lo - 0.15 * (hi - lo), hi + 0.15 * (hi - lo))
    fx = [r["focus"] for r in FOCUS if r["focus"] is not None]
    axes[0][0].set_xlim(min(fx) - 100, max(fx) + 100)
    axes[0][0].set_ylabel("focal length [px]")
    fig.tight_layout()
    savefig(fig, "zcam_focus_model.png")

### 5b  Mastcam-Z focus model with temperature, sol and principal point (v0p42)

The focal length of the backlash-state bins is fitted as f = f0 + a·(focus − 600) + b·(T_FPA + 15 °C) + c·(sol − 700), without per-scape offsets (a scape's offset would absorb its sol and temperature). Each block's Mastcam-Z focal lengths are first divided by its Navcam focal scale (refined / start), because the Mastcam-Z bins follow the Navcam scale through the shared points: in the 0.22 Three Forks block, Navcam was +0.35 % and Mastcam-Z was +0.40 %. The temperature is HEAD_FPA, which `attach_pds_labels` in §2a reads from the labels.

The **equivalent boresight** of each stereo pair is where the left principal point's ray lands in the right image: eqx = (cxR − cxL) + fR·yaw, eqy = (cyR − cyL) − fR·pitch. With a ~20° field, a principal point and the pointing trade off, so only this right-minus-left difference is observable. Its slope against focus is the principal-point slope of the model, put on ZR.

The cell writes a candidate `M2020_ZCAM034_focus_model.json` to `OUT/zcam_focus_model/`. To use it, set notebook 03's `ZCAM_FOCUS_MODEL` to its path. An existing project keeps its bins until it is rebuilt with `REPROCESS_ALL = True`.

In [ ]:
ZFIT, ZBORE = {}, {}
if FOCUS:
    ZFIT = {g: CAL.fit_focus_model(FOCUS, g, min_observations=MIN_OBS, min_focus=FOCUS_MIN, per_scape_offset=False,
                                   thermal=FOCUS_THERMAL, trend=FOCUS_TREND, navcam_normalise=FOCUS_NAVCAM_NORMALISE,
                                   reference_focus=600.0)
            for g in sorted({r["group"] for r in FOCUS})}
    for g, f in ZFIT.items():
        if not f:
            print(g, "too few bins (with temperatures) for the full fit")
            continue
        print(f"{g}: f = {f['f0_px']:.1f} + {f['slope_px_per_count']:.4f}±{f['slope_sd_px_per_count']:.4f} px/count"
              + (f"  {f['thermal']['f_px_per_degC']:+.3f}±{f['thermal']['sd_px_per_degC']:.3f} px/degC (T_FPA "
                 f"{f['thermal']['T_range_degC'][0]:.0f}..{f['thermal']['T_range_degC'][1]:.0f} C)" if f.get('thermal') else "")
              + (f"  {1e3 * f['trend']['f_px_per_sol']:+.1f}±{1e3 * f['trend']['sd_px_per_sol']:.1f} px/1000 sol" if f.get('trend') else "")
              + f"  rms {f['rms_px']:.2f} px, {f['n_bins']} bins")
    print("Navcam focal scale per scape:", {s: round(v, 5) for s, v in {r["scape"]: r["navcam_scale"] for r in FOCUS}.items() if v})
    BORE = CAL.zcam_boresight_table(sols)
    if BORE:
        ZBORE = CAL.fit_zcam_boresight(BORE, per_scape_offset=True, thermal=False)
        for q in ("eqx_px", "eqy_px", "roll_mdeg"):
            if q in ZBORE:
                b = ZBORE[q]
                print(f"R-L {q:9s}: {1e3 * b['slope_per_count']:+.2f}±{1e3 * b['sd']:.2f} per 1000 counts, scatter "
                      f"{b['scatter']:.2f}, {b['n']} pairs; per scape {({k: round(v, 2) for k, v in b['offsets'].items()})}")
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for ax, q in zip(axes, ("eqx_px", "eqy_px", "roll_mdeg")):
            for k, n_ in enumerate(sorted({r["scape"] for r in BORE})):
                rr = [r for r in BORE if r["scape"] == n_ and r["focus_left"] is not None]
                ax.scatter([0.5 * (r["focus_left"] + r["focus_right"]) for r in rr], [r[q] for r in rr], s=6, color=f"C{k}", label=n_)
            ax.set_xlabel("focus motor count"); ax.set_ylabel(q); ax.grid(alpha=0.3)
        axes[0].legend(fontsize=7); fig.tight_layout(); savefig(fig, "zcam_boresight_vs_focus.png")
    ok = {g: f for g, f in ZFIT.items() if f}
    if ok:
        model = CAL.focus_model_json(ok, ZBORE or None, source=f"MPPP {mppp.__version__} notebook 04 §5b: {', '.join(sols)}")
        (OUT / "zcam_focus_model").mkdir(parents=True, exist_ok=True)
        (OUT / "zcam_focus_model" / "M2020_ZCAM034_focus_model.json").write_text(json.dumps(model, indent=1, default=float), encoding="utf-8")
        print("candidate focus model:", OUT / "zcam_focus_model" / "M2020_ZCAM034_focus_model.json")

## 6  Mastcam-Z stereo pairs

Mastcam-Z has no rig in notebook 03: each image is its own frame, so every simultaneous left/right pair has its own refined relative pose. For each well-observed pair (≥ `MIN_PAIR_OBS` observations in each image), `zcam_pair_effects` gives the centre disparity error of two alternatives against that pair's refined geometry:

- **label**: the bins' starting cameras (label median) with the label relative pose.
- **median**: the pair's refined cameras with the median refined relative pose. This shows how well a single fixed Mastcam-Z rig would work.

In the **label** rows the rotation columns (`dyaw`, `dpitch`, `droll`) are measured from the label relative pose; in the **median** rows they are measured from the median rig. For Mastcam-Z at 34 mm, f·B ≈ 4700 px × 0.244 m ≈ 1150 px·m.

The principal point of each Mastcam-Z eye is held at the eye's label median, but the per-image label principal point varies (ZR034 at Rockytop spans 690–859 px in x), with the label attitude compensating. Calibrating the principal point would need a run with `zcam_bin_refine="all"`, or one principal point refined per eye.

In [ ]:
MIN_PAIR_OBS = 300
ZPAIRS = {}
for n_, s in sols.items():
    if not s.groups("Z"):
        continue
    for ref in ("label", "median"):
        ZPAIRS[(n_, ref)] = CAL.zcam_pair_effects(s, RANGES_M, MIN_PAIR_OBS, ref)
Z_SUM = []
for (n_, ref), rows in ZPAIRS.items():
    if not rows:
        continue
    a = lambda k: np.array([r[k] for r in rows], float)      # noqa: E731
    med = lambda v: float(np.median(v))                      # noqa: E731
    mad = lambda v: float(1.4826 * np.median(np.abs(v - np.median(v))))   # noqa: E731
    Z_SUM.append({"scape": n_, "reference": ref, "pairs": len(rows),
                  "dyaw_mdeg": med(a("dyaw_mdeg")), "dpitch_mdeg": med(a("dpitch_mdeg")), "droll_mdeg": med(a("droll_mdeg")),
                  "disp_10m_px": med(a("disparity_px_10m")), "disp_10m_sd_px": mad(a("disparity_px_10m")),
                  "range_10m_pct": med(a("range_error_pct_10m")), "range_10m_sd_pct": mad(a("range_error_pct_10m")),
                  "vparallax_10m_px": med(a("vertical_parallax_px_10m"))})
if Z_SUM:
    show(Z_SUM, None, 3)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    for (n_, ref), rows in ZPAIRS.items():
        if rows:
            axes[0 if ref == "label" else 1].scatter([r["focus_left"] for r in rows], [r["disparity_px_10m"] for r in rows],
                                                     s=8, alpha=0.6, label=n_)
    allv = np.array([r["disparity_px_10m"] for rows in ZPAIRS.values() for r in rows])
    lim = np.nanpercentile(allv, [1, 99]) + [-1, 1]
    for ax, t in zip(axes, ("label geometry", "one fixed (median) rig")):
        ax.set_ylim(*lim)
        ax.text(0.01, 0.02, f"{int(np.sum((allv < lim[0]) | (allv > lim[1])))} pairs off scale", transform=ax.transAxes,
                fontsize=7)
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xlabel("left focus count")
        ax.set_ylabel("centre disparity error at 10 m [px]")
        ax.set_title(f"Mastcam-Z pairs processed with the {t}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, "zcam_pair_disparity.png")

## 7  What limits the geometry

The table below puts the measured terms side by side, in pixels and as a range error at 10 m and 20 m, so each can be judged against the image precision ε (notebook 05: about 0.2–0.3 px, and about 0.3–0.4 px after the degrees-of-freedom correction).

- **Calibration repeatability** (Navcam, scape against consensus) bounds what a fixed camera model costs compared with per-scape refinement.
- **Label geometry** is the bias left if the flight calibration is used without refinement.
- **Fixed rig** (Mastcam-Z) is the cost of replacing per-pair poses with one rig.

A bias that is the same in every scape (the label rows) is a systematic error: it scales every range measurement by the same fraction. It can be removed by calibration, but not by averaging over scapes.

In [ ]:
PERF = []
fB = {"Navcam": 2956 * 0.4244, "Mastcam-Z": 4700 * 0.2435}
def rng(dpx, Z, fam):
    return 100 * Z * dpx / fB[fam]
for lens in sorted(lenses):
    rep = [r for r in DIFF if r["reference"] == f"consensus {lens}"]
    if rep:
        PERF.append({"camera": "Navcam", "term": f"intrinsics repeatability ({lens}), rms over frame",
                     "px": float(np.median([r["rms_px"] for r in rep])),
                     "px_worst": float(np.max([r["rms_px"] for r in rep]))})
    for which in ("consensus", "label", "initial"):
        rr = [r for r in STEREO if r["a"] == which and r["lens"] == lens and r["range_m"] == 10]
        if rr:
            v = np.array([r["centre_disparity_px"] for r in rr])
            PERF.append({"camera": "Navcam", "term": f"stereo disparity, {which} geometry ({lens})",
                         "px": float(np.median(v)), "px_worst": float(v[np.argmax(np.abs(v))]),
                         "range_10m_pct": rng(float(np.median(v)), 10, "Navcam"),
                         "range_20m_pct": rng(float(np.median(v)), 20, "Navcam")})
for r in Z_SUM:
    PERF.append({"camera": "Mastcam-Z", "term": f"stereo disparity, {r['reference']} geometry ({r['scape']})",
                 "px": r["disp_10m_px"], "px_worst": r["disp_10m_sd_px"],
                 "range_10m_pct": rng(r["disp_10m_px"], 10, "Mastcam-Z"), "range_20m_pct": rng(r["disp_10m_px"], 20, "Mastcam-Z")})
for g, f in (FITS.items() if FOCUS else []):
    if f:
        PERF.append({"camera": "Mastcam-Z", "term": f"{g} focal length: scape offsets sd (scale)",
                     "px": f["scape_offset_sd_px"], "range_10m_pct": 100 * f["scape_offset_sd_px"] / f["f0_px"],
                     "range_20m_pct": 100 * f["scape_offset_sd_px"] / f["f0_px"]})
        if f["label"]:
            d = f["f0_px"] - f["label"]["f0_px"]
            PERF.append({"camera": "Mastcam-Z", "term": f"{g} focal length: refined - label at ref focus (scale)",
                         "px": d, "range_10m_pct": 100 * d / f["f0_px"], "range_20m_pct": 100 * d / f["f0_px"]})
for s in sols.values():
    e = s.summary.get("eps") or {}
    if e.get("eps_intra_px"):
        PERF.append({"camera": "all", "term": f"image precision ε intra / cross ({s.label})",
                     "px": e["eps_intra_px"], "px_worst": e.get("eps_cross_px")})
show(PERF, ["camera", "term", "px", "px_worst", "range_10m_pct", "range_20m_pct"], 3)

**Reading the table.**

- `px` is the median over scapes or pairs. `px_worst` is the largest scape value; for the Mastcam-Z pair rows it is the pair-to-pair spread (1.48 × MAD).
- The stereo rows give the disparity error at the image centre, for a point at 10 m. The range error for any other range scales linearly with range (dZ/Z = Z·Δd/(f·B)).
- For Mastcam-Z focal length, the scale error applies directly to ranges from Mastcam-Z stereo.
- The ε rows are the post-fit image precision, for comparison: a term far below ε is negligible, and a term above ε is worth calibrating out.

## 8  Updated calibration: CAHVORE (Navcam) and CAHVOR (Mastcam-Z)

The consensus refined cameras are written back as flight-style models.

**Navcam.** CAHVORE of type 2 (fisheye, the label's type) and type 3 (general, with a fitted linearity), in the left camera frame (C_NL = 0). NR is placed by the mean refined rig. E is zero because COLMAP cameras are central. The fit residual shows how well each model type represents the refined camera.

**Mastcam-Z.** CAHVOR models from the focal-length fit at a few focus counts. The principal point and distortion are the held label-median values, and fx/fy is the median refined aspect.

Placing these in the rover frame for an image needs that image's pose. `CameraModel.in_frame` and `mppp.cmod.model_in_world` do this with the pose from the refined solution. The label's C and A can be kept if only the intrinsics are to be updated.

In [ ]:
UPD = {}
lens = "rational" if ("NL", "rational") in NAV else (sorted(lenses)[0] if lenses else None)
if lens and ("NL", lens) in NAV and ("NR", lens) in NAV:
    rig = CAL.mean_rig(sols, lens)
    for mt in (2, 3):
        u = CAL.updated_navcam_models(NAV[("NL", lens)], NAV[("NR", lens)], rig, mtype=mt)
        UPD[f"navcam_type{mt}"] = u
        print(f"CAHVORE type {mt}: fit rms NL {u['fit_left']['rms_px']:.3f} px (corners {u['fit_left']['corner_rms_px']:.2f}),"
              f" NR {u['fit_right']['rms_px']:.3f} px (corners {u['fit_right']['corner_rms_px']:.2f})")
    print(UPD["navcam_type2"]["text"])
    # the updated pair against the label pair: intrinsics and stereo
    for g, key in (("NL", "left"), ("NR", "right")):
        lab = next((m for m in (CAL.median_label_model(s, g) for s in sols.values() if s.manifest) if m is not None), None)
        if lab is not None:
            d = compare_cameras(PixelCamera.cahv(lab, 5120, 3840, as_is=True),
                                PixelCamera.cahv(UPD["navcam_type2"][key], 5120, 3840), step=96)
            print(f"{g}: updated CAHVORE vs label: rms {d['rms_px']:.2f} px, centre {d['centre_rms_px']:.2f}, "
                  f"corners {d['corner_rms_px']:.2f}, rotation {d['rotation_deg']:.3f} deg")
else:
    print("no refined Navcam consensus")

if FOCUS:
    ZUPD = CAL.updated_zcam_models(sols, {g: f for g, f in FITS.items() if f}, focus_counts=(-1000, 0, 500, 1000))
    for g, rows in ZUPD.items():
        for r in rows:
            print(f"/* {g} at focus {r['focus']:+.0f}: f = {r['f_px']:.1f} px (fit rms {r['fit_rms_px']:.1e} px) */")
            print(r["text"])
    UPD["zcam"] = ZUPD

def _jsonable(u):
    out = {}
    for k, v in u.items():
        if k == "zcam":
            out[k] = {g: [{"focus": r["focus"], "f_px": r["f_px"], "model": r["model"].to_label_dict(12)} for r in rows]
                      for g, rows in v.items()}
        else:
            out[k] = {"frame": "NL camera frame, full-frame pixels, pixel-centre origin",
                      "NL": v["left"].to_label_dict(12), "NR": v["right"].to_label_dict(12),
                      "fit_NL": {a: b for a, b in v["fit_left"].items()}, "fit_NR": {a: b for a, b in v["fit_right"].items()}}
    return out
(OUT / "updated_camera_models.json").write_text(json.dumps(_jsonable(UPD), indent=1, default=float), encoding="utf-8")
(OUT / "updated_camera_models.txt").write_text(
    "\n\n".join([UPD[k]["text"] for k in ("navcam_type2", "navcam_type3") if k in UPD] +
                [f"/* {g} at focus {r['focus']:+.0f}: f = {r['f_px']:.2f} px */\n{r['text']}"
                 for g, rows in UPD.get("zcam", {}).items() for r in rows]), encoding="utf-8")
print("saved", OUT / "updated_camera_models.json", "and .txt")

## 9  Example images

For each camera, one image is shown twice: as recorded (distorted) and resampled to a pinhole camera with that image's refined lens model. The hardware mask (mask = 0 over valid pixels) carries a white screen of opacity `SCREEN_ALPHA`, so the rover stays visible. A grid that is straight in the pinhole image is drawn through the lens model on the original.

The image with the most tie points is chosen from the first scape that has the camera. Full-frame images, covering the whole detector at full resolution, are preferred: many Navcam products are sub-frames or downsampled. `EXAMPLE_FILES` overrides this with an image file of your own (and optionally a mask) for any camera. The camera model is then the refined model of `EXAMPLE_SCAPE[camera]`.

In [ ]:
for g in ("NL", "NR", "ZL034", "ZR034"):
    name = EXAMPLE_SCAPE.get(g) or next((n_ for n_, s in sols.items() if g in s.groups()), None)
    if name is None:
        continue
    s = sols[name]
    if g in EXAMPLE_FILES:
        ip, mp = (tuple(EXAMPLE_FILES[g]) + (None,))[:2]
        nm = Path(ip).name
        # an image of this scape uses its own refined camera (Mastcam-Z: its focus bin)
        cam = s.camera_of_image(nm) if nm in s.images else max(s.cameras_of(g), key=lambda c: c.n_obs)
        title = f"{name}: {nm}" + ("" if nm in s.images else f" (model {cam.key})")
        img, mask = CAL.load_example(s, nm, ip, mp)
    else:
        img_name = CAL.pick_example(s, g)
        cam = s.camera_of_image(img_name)
        title = f"{name}: {img_name}"
        try:
            img, mask = CAL.load_example(s, img_name)
        except FileNotFoundError as e:
            print(g, "image not found:", e); continue
    fig, und = CAL.example_figure(img, mask, cam, title=title, alpha=SCREEN_ALPHA)
    savefig(fig, f"example_{g}.png")
    plt.show()

## 10  Save

In [ ]:
def _clean(rows):
    return [{k: v for k, v in r.items() if not k.startswith("_") and not isinstance(v, np.ndarray)} for r in rows]
report = {"mppp_version": mppp.__version__, "scapes": {k: str(v) for k, v in SCAPES.items()},
          "solutions": CAL.solutions_table(sols), "navcam_differences": _clean(DIFF), "label_vs_refined": _clean(LABEL_VS),
          "navcam_rig": _clean(RIG), "navcam_stereo": _clean(STEREO), "zcam_focus": FOCUS,
          "zcam_focus_fits": FITS if FOCUS else {},
          "zcam_focus_model_fit": ZFIT, "zcam_boresight": ZBORE, "zcam_pairs": _clean(Z_SUM), "performance": _clean(PERF)}
(OUT / "camera_models_report.json").write_text(json.dumps(report, indent=1, default=float), encoding="utf-8")
print("saved", OUT / "camera_models_report.json")